# Joint-flow alpha inference

Use a `flow_id` whose name already encodes the KIC weight/query choice.

In [ ]:
import numpyro, jax
num_chains = 4
numpyro.set_host_device_count(num_chains)
print ('# jax device count:', jax.local_device_count())
from pathlib import Path
import sys

cwd = Path.cwd()
LIKELIHOOD_DIR = cwd if cwd.name == "likelihood_wkic_joint" else cwd / "likelihood_wkic_joint"
REPO_ROOT = LIKELIHOOD_DIR.parent
for path in (REPO_ROOT, LIKELIHOOD_DIR):
    if str(path) not in sys.path:
        sys.path.insert(0, str(path))

from likelihood_wkic_joint.alpha_inference import run_and_save_alpha_inference


In [ ]:
P_SHORT_QUERY = "koi_period < 8.6"
P_LONG_QUERY = "koi_period > 8.6"

BASE_SPEC = dict(
    n_grid=20,
    num_warmup=1000,
    num_samples=1000,
    num_chains=num_chains,
    rng_seed=0,
    svi_seed=0,
    x_index=0,
    shift_index=1,
    alpha_mean_prior="uniform",
    alpha_mean_bounds=(-0.2, 0.2),
    ylabel=r"$r_\mathrm{KOI}-r_\mathrm{KIC}$",
    max_tree_depth=10,
    target_accept_prob=0.9,
    svi_step_size=1e-1,
    svi_num_steps=4000,

)

SPECS_BY_FLOW_ID = {
    "m15_kic_pdet_teff2_all": dict(
        BASE_SPEC,
        alpha_name="alpha_teff",
        flow_id="m15_kic_pdet_teff2_all",
        x_name="Teff",
        x_label=r"$T_\mathrm{eff}$ (K)",
        koi_query=None,
    ),
    "m15_kic_pdet_teff5_all": dict(
        BASE_SPEC,
        alpha_name="alpha_teff",
        flow_id="m15_kic_pdet_teff5_all",
        x_name="Teff",
        x_label=r"$T_\mathrm{eff}$ (K)",
        koi_query=None,
    ),
    "m15_kic_pdet_teff5_pshort": dict(
        BASE_SPEC,
        alpha_name="alpha_teff",
        flow_id="m15_kic_pdet_teff5_pshort",
        x_name="Teff",
        x_label=r"$T_\mathrm{eff}$ (K)",
        koi_query=P_SHORT_QUERY,
    ),
    "m15_kic_pdet_teff5_plong": dict(
        BASE_SPEC,
        alpha_name="alpha_teff",
        flow_id="m15_kic_pdet_teff5_plong",
        x_name="Teff",
        x_label=r"$T_\mathrm{eff}$ (K)",
        koi_query=P_LONG_QUERY,
    ),
    "kic_pdet_rossby5_all": dict(
        BASE_SPEC,
        alpha_name="alpha_logro",
        flow_id="kic_pdet_rossby5_all",
        x_name="logRo",
        x_label=r"$\log\mathrm{Ro}$",
        koi_query=None,
    ),
    "kic_pdet_rossby5_pshort": dict(
        BASE_SPEC,
        alpha_name="alpha_logro",
        flow_id="kic_pdet_rossby5_pshort",
        x_name="logRo",
        x_label=r"$\log\mathrm{Ro}$",
        koi_query=P_SHORT_QUERY,
    ),
    "kic_pdet_rossby5_plong": dict(
        BASE_SPEC,
        alpha_name="alpha_logro",
        flow_id="kic_pdet_rossby5_plong",
        x_name="logRo",
        x_label=r"$\log\mathrm{Ro}$",
        koi_query=P_LONG_QUERY,
    ),
}

FLOW_IDS_TO_RUN = [
    "m15_kic_pdet_teff2_all",
    "m15_kic_pdet_teff5_all",
    "m15_kic_pdet_teff5_pshort",
    "m15_kic_pdet_teff5_plong",
]
SPECS = [SPECS_BY_FLOW_ID[flow_id] for flow_id in FLOW_IDS_TO_RUN]
FLOW_IDS_TO_RUN


In [ ]:
runs = {}
for flow_id in FLOW_IDS_TO_RUN[:2]:
    spec = dict(SPECS_BY_FLOW_ID[flow_id])
    print(flow_id)
    print(f"  alpha: {spec['alpha_name']}")
    print(f"  koi_query: {spec['koi_query']}")
    result = run_and_save_alpha_inference(spec)
    runs[flow_id] = result["paths"]
    print(f"  saved -> {result['paths']['output_dir']}")

runs


In [ ]:
for flow_id in FLOW_IDS_TO_RUN[2:]:
    spec = dict(SPECS_BY_FLOW_ID[flow_id])
    print(flow_id)
    print(f"  alpha: {spec['alpha_name']}")
    print(f"  koi_query: {spec['koi_query']}")
    result = run_and_save_alpha_inference(spec)
    runs[flow_id] = result["paths"]
    print(f"  saved -> {result['paths']['output_dir']}")